# Appendix 10.2: Tool Use

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `chat` helper function.

In [ ]:
!pip install anthropic

import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def chat(messages, system_prompt="", tools=None, tool_choice=None):
    params = {
        "model": MODEL_NAME,
        "max_tokens": 2000,
        "temperature": 0.0,
        "system": system_prompt,
        "messages": messages,
    }
    if tools:
        params["tools"] = tools
    if tool_choice:
        params["tool_choice"] = tool_choice
    return client.messages.create(**params)

---

## Lesson

**Tool use** (also called function calling) lets Claude call functions you define — a calculator, a database query, a weather API, your own internal service — and use the results to answer the user. Claude never literally executes your code. Instead, the flow is:

1. You describe your tool(s) to Claude with a JSON Schema, via the `tools` parameter.
2. Claude decides whether it needs a tool to answer. If so, it stops generating text and returns a `tool_use` content block (`stop_reason` will be `"tool_use"`) naming the tool and the arguments it wants to call it with.
3. **You** run the actual function with those arguments, then send the result back to Claude as a `tool_result` content block in a new `user` turn.
4. Claude reads the result and continues — either calling another tool or giving you a final text answer.

This is a structured, first-class part of the Messages API today. (Older versions of this course taught a manual, XML-based version of this same idea — hand-writing a system prompt describing tools, asking Claude to emit `<function_calls>` XML, and parsing it yourself with regex. That approach is obsolete; the mechanics below replace it entirely and are far more reliable.)

Tool use expands Claude's capabilities and lets it handle much more complex, multi-step, *agentic* tasks.
Some examples of tools you can give Claude:
- Calculator
- Word counter
- SQL database querying and data retrieval
- Weather API

Anthropic also now ships several **server tools** — web search, code execution, a bash shell, a text editor — that Claude can use directly with zero implementation work from you (Anthropic runs them, not your code). This appendix focuses on **client-side custom tools** like the ones above, since that's the pattern you'll use for anything specific to your own systems. See [the tool use docs](https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/overview) for the full list of server tools.

You get Claude to use tools by combining two elements:

1. **Tool definitions** — a list of tools passed via the `tools` parameter, each with a `name`, a `description` (this is what Claude actually reasons over to decide *when* and *how* to call the tool — write it like you're briefing a new engineer), and an `input_schema` (standard JSON Schema).
2. **Control logic** — the code on your side that inspects Claude's response, runs the requested tool(s), and feeds the result(s) back.

### What's new since the old XML-based format

* Tool calls and results are **structured content blocks** (`tool_use` / `tool_result`), not hand-parsed XML.
* **Parallel tool calls** — a single response can contain multiple `tool_use` blocks at once (e.g. "what's the weather in Paris and Tokyo?" triggers two independent calls Claude expects to run concurrently).
* **`tool_choice`** gives you explicit control: let Claude decide (`auto`), force it to use *some* tool (`any`), force one specific tool (`tool`), or forbid tool use entirely (`none`).
* **Token-efficient tool use** and fine-grained streaming reduce the latency/cost overhead of tool calls.
* The same `tools` + `input_schema` mechanism is also the modern way to force **structured JSON output** — see Appendix 10.6.

### Examples

To enable tool use, we define our tool(s) with a JSON Schema and pass them in the `tools` parameter — no special system prompt boilerplate required. Here's a simple calculator tool.

In [ ]:
calculator_tool = {
    "name": "calculator",
    "description": "A simple calculator that performs one arithmetic operation between two numbers. Use this any time you need an exact numeric result rather than an estimate.",
    "input_schema": {
        "type": "object",
        "properties": {
            "num1": {"type": "number", "description": "The first operand."},
            "num2": {"type": "number", "description": "The second operand."},
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform."
            }
        },
        "required": ["num1", "num2", "operation"]
    }
}

Now let's give Claude a question that requires this tool.

In [ ]:
messages = [{"role": "user", "content": "Multiply 1,984,135 by 9,343,116"}]

response = chat(messages, tools=[calculator_tool])

print("stop_reason:", response.stop_reason)
print("content:", response.content)

`stop_reason` is `"tool_use"`, and `content` holds a `tool_use` block with an `id`, the tool `name`, and the parsed `input` — already valid according to our schema, no manual XML parsing required.

Now let's actually run the calculator and send the result back. First, the function itself:

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == "add":
        return num1 + num2
    elif operation == "subtract":
        return num1 - num2
    elif operation == "multiply":
        return num1 * num2
    elif operation == "divide":
        return num1 / num2
    else:
        raise ValueError(f"Unsupported operation: {operation}")

Next, pull the `tool_use` block out of Claude's response, run the function, and package the result as a `tool_result` block. Then we append **both** Claude's tool-call turn and our tool-result turn to the message history and call Claude again so it can finish answering.

In [ ]:
tool_use_block = next(block for block in response.content if block.type == "tool_use")
result = do_pairwise_arithmetic(**tool_use_block.input)

messages.append({"role": "assistant", "content": response.content})
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": str(result)
        }
    ]
})

final_response = chat(messages, tools=[calculator_tool])
print(final_response.content[0].text)

Congratulations on running an entire tool use round trip end to end! Now let's see what happens when a question **doesn't** require the tool at all.

In [ ]:
messages = [{"role": "user", "content": "Tell me the capital of France."}]

response = chat(messages, tools=[calculator_tool])
print("stop_reason:", response.stop_reason)
print(response.content[0].text)

Success — `stop_reason` is `"end_turn"`, not `"tool_use"`, and Claude just answers directly. Claude decides *whether* to call a tool based on the question and your tool descriptions; you don't have to tell it when not to bother.

### Parallel tool calls

When a request naturally decomposes into independent sub-calls, Claude can return **multiple `tool_use` blocks in a single response** instead of one at a time. Your control logic should always loop over *all* tool_use blocks in `response.content` and return a `tool_result` for each — in the same `user` turn — rather than assuming there's exactly one.

In [ ]:
messages = [{"role": "user", "content": "What's 12 times 8? Also, what's 100 divided by 4?"}]

response = chat(messages, tools=[calculator_tool])
tool_uses = [block for block in response.content if block.type == "tool_use"]
print(f"Claude made {len(tool_uses)} tool call(s) in this turn:")
for block in tool_uses:
    print(" -", block.name, block.input)

### `tool_choice`

By default (`{"type": "auto"}`) Claude decides whether and which tool to use. You can be more directive:
- `{"type": "any"}` — Claude must call *some* tool, but you don't care which.
- `{"type": "tool", "name": "calculator"}` — Claude must call this specific tool.
- `{"type": "none"}` — Claude must not call any tool, even if one is provided (useful when you want to see what Claude would say in plain text without removing the tool definitions from the request).

Forcing `any` or a specific `tool` is handy in exercises and tests where you want to guarantee a structured response instead of Claude occasionally deciding to just answer in prose.

In [ ]:
messages = [{"role": "user", "content": "Multiply 7 by 6"}]

response = chat(messages, tools=[calculator_tool], tool_choice={"type": "tool", "name": "calculator"})
print(response.stop_reason)
print(response.content)

If you would like to experiment with the lesson prompts without changing any content above, scroll all the way to the bottom of the lesson notebook to visit the [**Example Playground**](#example-playground).

---

## Exercises
- [Exercise 10.2.1 - SQL](#exercise-1021---sql)

### Exercise 10.2.1 - SQL
In this exercise, you'll write tool definitions for querying and writing to the world's smallest "database". Here's the initialized database, which is really just a dictionary.

In [ ]:
db = {
    "users": [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"},
        {"id": 3, "name": "Charlie", "email": "charlie@example.com"}
    ],
    "products": [
        {"id": 1, "name": "Widget", "price": 9.99},
        {"id": 2, "name": "Gadget", "price": 14.99},
        {"id": 3, "name": "Doohickey", "price": 19.99}
    ]
}

And here is the code for the functions that read and write the database.

In [ ]:
def get_user(user_id):
    for user in db["users"]:
        if user["id"] == user_id:
            return user
    return None

def get_product(product_id):
    for product in db["products"]:
        if product["id"] == product_id:
            return product
    return None

def add_user(name, email):
    user_id = len(db["users"]) + 1
    user = {"id": user_id, "name": name, "email": email}
    db["users"].append(user)
    return user

def add_product(name, price):
    product_id = len(db["products"]) + 1
    product = {"id": product_id, "name": name, "price": price}
    db["products"].append(product)
    return product

To solve the exercise, define a `tools` list containing one tool definition per function above (`get_user`, `get_product`, `add_user`, `add_product`), each with a name, a clear description, and an `input_schema`. We've given you the scaffolding and a dispatcher below — fill in `tools`.

In [ ]:
# Your tool definitions go here — this is the only field you should change
tools = []

def execute_tool(name, tool_input):
    if name == "get_user":
        return get_user(tool_input["user_id"])
    elif name == "get_product":
        return get_product(tool_input["product_id"])
    elif name == "add_user":
        return add_user(tool_input["name"], tool_input["email"])
    elif name == "add_product":
        return add_product(tool_input["name"], tool_input["price"])
    else:
        return f"Unknown tool: {name}"

When you're ready, run your tool definitions against the examples below. This harness loops until Claude stops asking for tools, running every tool call it requests (including parallel ones) along the way — you shouldn't need to modify it.

In [ ]:
def run_tool_loop(prompt, tools):
    messages = [{"role": "user", "content": prompt}]
    response = chat(messages, tools=tools)
    while response.stop_reason == "tool_use":
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        messages.append({"role": "user", "content": tool_results})
        response = chat(messages, tools=tools)
    return next((b.text for b in response.content if b.type == "text"), "")

examples = [
    "Add a user to the database named Deborah.",
    "Add a product to the database named Thingo",
    "Tell me the name of User 2",
    "Tell me the name of Product 3"
]

for example in examples:
    print(example, "\n----------\n", run_tool_loop(example, tools), "\n*********\n")

If you did it right, Claude should correctly call `add_user`, `add_product`, `get_user`, and `get_product` as needed. Print `db` afterward to see the "database" reflect the new rows.

❓ If you want to see a possible solution, run the cell below!

In [ ]:
from hints import exercise_10_2_1_solution; print(exercise_10_2_1_solution)

### Congrats!
Congratulations on learning tool use! Head over to Appendix 10.3 for search & retrieval, or skip ahead to Appendix 10.4-10.9 to learn about prompt caching, extended thinking, structured outputs, citations, computer use / agents, and MCP — all of which build directly on the tool use concepts from this appendix.

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect Claude's responses.

In [ ]:
calculator_tool = {
    "name": "calculator",
    "description": "A simple calculator that performs one arithmetic operation between two numbers. Use this any time you need an exact numeric result rather than an estimate.",
    "input_schema": {
        "type": "object",
        "properties": {
            "num1": {"type": "number", "description": "The first operand."},
            "num2": {"type": "number", "description": "The second operand."},
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform."
            }
        },
        "required": ["num1", "num2", "operation"]
    }
}

def do_pairwise_arithmetic(num1, num2, operation):
    if operation == "add":
        return num1 + num2
    elif operation == "subtract":
        return num1 - num2
    elif operation == "multiply":
        return num1 * num2
    elif operation == "divide":
        return num1 / num2
    else:
        raise ValueError(f"Unsupported operation: {operation}")

In [ ]:
messages = [{"role": "user", "content": "Multiply 1,984,135 by 9,343,116"}]

response = chat(messages, tools=[calculator_tool])
print("stop_reason:", response.stop_reason)
print("content:", response.content)

In [ ]:
tool_use_block = next(block for block in response.content if block.type == "tool_use")
result = do_pairwise_arithmetic(**tool_use_block.input)

messages.append({"role": "assistant", "content": response.content})
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": str(result)
        }
    ]
})

final_response = chat(messages, tools=[calculator_tool])
print(final_response.content[0].text)

In [ ]:
messages = [{"role": "user", "content": "Tell me the capital of France."}]

response = chat(messages, tools=[calculator_tool])
print("stop_reason:", response.stop_reason)
print(response.content[0].text)